# Baltimore's Renaissance: Explainer Notebook
### 02806 Social Data Analysis and Visualization: DTU Spring 2026
**Benjamin Balslund (s253848), Design & Innovation**

This notebook documents the data analysis behind the website
[Baltimore's Renaissance: How a City Stopped Counting Its Dead](#).
The website uses crime data, demographic indicators, and eviction statistics
to tell the story of Baltimore's historic decline in homicides since 2020.

All code is reproducible. Primary dataset: Open Baltimore Part 1 Crime Data,
499,059 records, 1 January 2014 to 28 December 2024.


## 1. Motivation

### What is the dataset?
The primary dataset is the **Open Baltimore Part 1 Crime Data**, a publicly available
record of all reported Part 1 crimes in Baltimore City from 2014 to 2024.
It contains 499,059 rows and 23 columns, covering 13 crime categories including
homicide, shooting, robbery, burglary, and larceny. Each record includes a timestamp,
crime type, weapon used, victim age and gender, and GPS coordinates.

Secondary datasets used:
- **Annie E. Casey Foundation KIDS COUNT**: children in single-mother households
  and teens not in school or work, Baltimore City 2014-2024
- **FRED / U.S. Census Bureau ACS**: disconnected youth (16-24) percentage, 2020-2024
- **FBI Uniform Crime Reports via city-data.com**: Baltimore police officer counts
- **Baltimore Neighborhood Indicators Alliance**: eviction rate per 1,000 residents by CSA, 2023
- **CDC via Pew Research Center**: US gun homicides 2014-2024
- **Baltimore City Mayor's Office**: 2025 homicide figures

### Why this dataset?
In late 2024, Channel 5 released *Baltimore Streets*, a documentary by Andrew Callaghan
exploring how Baltimore's homicide rate had begun to decline dramatically. The documentary
raised questions this dataset could actually answer: Is the decline real? Where is it
happening? What structural factors correlate with it?

As a Design & Innovation student, I was interested in how data visualisation could tell
a human story, not just confirm statistics, but connect numbers to the lived experience
of the people in the documentary.

### Goal for the end user
The website targets a non-specialist reader, someone who has seen the documentary
or is curious about Baltimore, but has no background in data analysis. The goal is
to make the data legible and emotionally meaningful: to show that the numbers confirm
what people on the ground are saying, while being honest about the limits of what
the data can prove.


## 2. Basic Stats

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load primary dataset
df = pd.read_excel('Part1_Crime_Beta_-7221791009442168097.xlsx', engine='openpyxl')
df['CrimeDateTime'] = pd.to_datetime(df['CrimeDateTime'])
df['Year'] = df['CrimeDateTime'].dt.year
df['Age']  = pd.to_numeric(df['Age'], errors='coerce')

print(f"Shape: {df.shape}")
print(f"Date range: {df['CrimeDateTime'].min().date()} to {df['CrimeDateTime'].max().date()}")
print(f"\nCrime types:")
print(df['Description'].value_counts())


In [ ]:
# Data quality check
print("Missing values (%):")
missing = (df.isnull().sum() / len(df) * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False))


### Data cleaning decisions

Several columns have substantial missing data:
- **Weapon** (70.8% missing): most crimes do not have a weapon recorded. For homicide
  and shooting analysis this is less of an issue since we filter by Description.
- **Age** (significant missings): used only for the 16-19 youth analysis; rows with
  missing age are excluded for that figure only.
- **Neighborhood** (1.3% missing): excluded from neighbourhood-level counts.
- **New_District** and **Old-District** was merged.

No imputation was performed. Missing values are excluded from the specific analyses
that require them, and this is noted in each figure's caption on the website.

The documentary cites a "54% reduction since 2020". Our dataset shows a **40% reduction**
(332 homicides in 2020 to 199 in 2024). The discrepancy likely reflects that the
documentary uses 2025 data (133 homicides) relative to a different baseline year.
This is noted explicitly in the data callout on the website.


In [ ]:
# Annual homicide counts — the headline figure
homicides = df[df['Description'] == 'HOMICIDE'].groupby('Year').size()
print("Annual homicides:")
print(homicides)
print(f"\nPeak year: {homicides.idxmax()} ({homicides.max()})")
print(f"Latest year: 2024 ({homicides[2024]})")
print(f"Change 2020-2024: {((homicides[2024] - homicides[2020]) / homicides[2020] * 100):.1f}%")


In [ ]:
# Crime type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: crime type breakdown
crime_counts = df['Description'].value_counts()
axes[0].barh(crime_counts.index, crime_counts.values, color='#E63946', alpha=0.8)
axes[0].set_xlabel('Incidents')
axes[0].set_title('Crime Type Distribution (2014-2024)')
axes[0].invert_yaxis()

# Right: homicide trend
years = list(range(2014, 2025))
hom_counts = [df[(df['Description']=='HOMICIDE') & (df['Year']==y)].shape[0] for y in years]
axes[1].plot(years, hom_counts, color='#E63946', linewidth=2.5, marker='o', markersize=6)
axes[1].axhline(300, color='#AAAAAA', linestyle=':', linewidth=1.5, label='300 threshold')
axes[1].axvline(2020, color='#2A9D8F', linestyle='--', linewidth=2, label='Mayor Scott elected')
axes[1].set_title('Annual Homicides 2014-2024')
axes[1].set_ylabel('Homicides')
axes[1].legend()
axes[1].set_xticks(years)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")


## 3. Data Analysis

### 3.1 Geographic concentration of violence
A key finding from the EDA is that gun violence is highly geographically concentrated.
The 10 Safe Streets zones, covering a small fraction of Baltimore's 278 neighbourhoods,
account for a disproportionate share of homicides and shootings every year.


In [ ]:
# Safe Streets zones and their homicide counts
ss_zones = ['SANDTOWN-WINCHESTER', 'BROOKLYN', 'BELAIR-EDISON', 'CHERRY HILL',
            'FRANKLIN SQUARE', 'MCELDERRY PARK', 'CENTRAL PARK HEIGHTS',
            'PENN NORTH', 'BELVEDERE', 'WOODBOURNE-MCCABE']

hom = df[df['Description'] == 'HOMICIDE'].copy()
hom['Neighborhood_upper'] = hom['Neighborhood'].str.upper()

ss_hom = hom[hom['Neighborhood_upper'].isin(ss_zones)]
total_hom = len(hom.dropna(subset=['Neighborhood']))

print(f"Total homicides with neighbourhood: {total_hom}")
print(f"Homicides in Safe Streets zones: {len(ss_hom)}")
print(f"Percentage: {len(ss_hom)/total_hom*100:.1f}%")
print()
print("By zone (all years):")
print(ss_hom['Neighborhood_upper'].value_counts())


### 3.2 Youth gun violence

The documentary focuses heavily on young people. The dataset includes victim age,
which allows us to isolate gun violence (homicide + shooting) involving 16-19 year olds.


In [ ]:
# Youth gun violence (16-19) — homicide + shooting
gun_crimes = df[df['Description'].isin(['HOMICIDE', 'SHOOTING'])].copy()
youth = gun_crimes[gun_crimes['Age'].between(16, 19)]

youth_by_year = youth.groupby('Year').size()
print("Gun violence involving 16-19 year olds:")
print(youth_by_year)
print(f"\nPeak: {youth_by_year.idxmax()} ({youth_by_year.max()} incidents)")
print(f"2024: {youth_by_year[2024]} incidents")
print(f"Change 2019-2024: {((youth_by_year[2024]-youth_by_year[2019])/youth_by_year[2019]*100):.1f}%")


**Note on age field:** The `Age` column in the dataset most likely refers to the
victim rather than an offender, based on the mean age for homicide victims (31.8 years),
which is consistent with external sources. However, this is not explicitly documented
in the Open Baltimore data dictionary, and all age-based findings on the website
carry a transparency note to this effect.


### 3.3 Correlation analysis: disconnected youth and gun violence

The website shows disconnected youth percentage alongside youth gun violence.
A natural question is whether these two series are statistically correlated.


In [ ]:
from scipy.stats import pearsonr, spearmanr

years = [2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
gun_youth  = [52, 101, 105, 120, 95, 139, 119, 96, 126, 149, 71]

# FRED/ACS: 2020-2024 precise, 2014-2019 estimated from trend
disconnected = [14.5, 14.0, 13.0, 12.5, 11.5, 10.5, 10.9, 10.1, 9.8, 8.6, 9.0]

r, p = pearsonr(disconnected, gun_youth)
rs, ps = spearmanr(disconnected, gun_youth)

print(f"Pearson r  = {r:.3f},  p = {p:.3f}")
print(f"Spearman r = {rs:.3f}, p = {ps:.3f}")
print(f"N = {len(years)}")
print()
print("Interpretation:")
print(f"  Negative r ({r:.2f}) = both series decline together, as expected.")
print(f"  p = {p:.3f} > 0.05: not statistically significant at N={len(years)}.")
print(f"  Would need r > 0.60 or N > 20 to reach significance.")
print()
print("Caveat: 2014-2019 disconnected youth values are estimated, not observed.")
print("This further limits the strength of any statistical claim.")


### 3.4 Note on machine learning

This project does not use machine learning. The decision was deliberate:
the primary analytical goal is descriptive and narrative, not predictive.
The dataset spans 11 years with a clear structural break in 2020-2023,
making time-series prediction methodologically challenging without additional
covariates. A clustering analysis of neighbourhoods by crime profile was considered
but excluded because the Safe Streets zone selection already represents an
evidence-based geographic segmentation from the city itself.

What the analysis does instead is use correlation, descriptive statistics, and
geographic visualisation to identify patterns and present them honestly,
including explicitly noting where causal claims cannot be supported.


## 4. Genre

Following Segel and Heer (2010), *Narrative Visualization: Telling Stories with Data*,
this project uses the **Martini Glass** structure: a directed narrative in the first half
(the author guides the reader through the story) that opens into an interactive exploration
at the centre (Figure 3, the dual slideshow map), before returning to a directed conclusion.

### Visual Narrative tools (Figure 7, Segel & Heer)

**Visual Structuring:**
- *Consistent visual platform*: all figures use the same colour palette (red for violence,
  teal for the 2020 turning point, amber for structural indicators), typography
  (Source Serif 4, Playfair Display, JetBrains Mono), and layout.
- *Progress bar*: not used. The page is a single scrolling narrative.
- *Introductory text*: each section opens with a video timestamp and contextual paragraph
  before presenting data.

**Highlighting:**
- *Feature distinction*: Safe Streets zones are consistently highlighted with coloured
  polygons across all map figures. The 2020 teal vertical line appears in every time
  series figure.
- *Annotations*: key data points are annotated directly on figures (2023 first below 300,
  2024 lowest on record, ACS anomaly in 2016).

**Transition guidance:**
- *Animated transitions*: Figure 3 uses Plotly.react() for smooth year-to-year transitions.
- *Bridging text*: every figure is preceded by a "↓ what the data shows" bridge element
  and followed by interpretive text.

### Narrative Structure tools (Figure 7, Segel & Heer)

**Ordering:**
- *User directed path*: the page is linear scroll, but Figure 3 allows the user to
  step through years independently. Figure 5 (eviction heatmap) and Figure 6 allow
  hover exploration.

**Interactivity:**
- *Hover highlighting*: all Plotly figures support hover tooltips.
- *Filtering/selection*: Figure 3 allows year selection via buttons.
- *Navigation buttons*: Figure 3 prev/next controls.

**Messaging:**
- *Captions*: every figure has a detailed caption explaining the data source, methodology,
  and interpretive caveats.
- *Annotations*: used to flag key events (Mayor Scott elected, ACS anomaly, first year
  below 300 homicides).
- *Summary*: the final section "The Renaissance is Real" summarises the full narrative arc.


## 5. Visualizations

### Figure 1: Annual homicide trend (line chart)
A simple line chart of homicides per year with a 300-threshold reference line.
Chosen because the central claim of the story — that Baltimore averaged over 300
homicides for years and then broke that streak — is most clearly communicated as
a time series. The teal vertical line at 2020 is used consistently across all figures
to anchor the Mayor Scott turning point visually.

### Figure 2: Police officers vs homicides (dual-axis bar/line)
Dual-axis because the two series have different units and scales. The grey bars
for officer count intentionally recede visually so the red homicide line is the
focal element. The point is not that fewer officers caused fewer homicides, but
that the assumption "more police = less crime" is not supported by this data.

### Figure 3: Dual slideshow map (scattermapbox + choroplethmapbox)
The most complex figure. Two side-by-side Mapbox maps showing homicide and shooting
locations for each year, with Safe Streets zone polygons drawn from official NSA
boundary GeoJSON. Year navigation via buttons uses Plotly.react() to update traces
without re-rendering the map. This is the primary interactive/exploratory element
of the site — it allows the reader to see the geographic persistence of violence
in specific neighbourhoods and trace the decline year by year.

### Figure 4a: Youth gun violence (bar chart)
Deliberately simple. The story is about young people, and a clean red bar chart
communicates the scale and trend without distraction.

### Figure 4b: Disconnected youth and single-mother households (dual-axis line)
Two structural indicators on the same chart to show they move together over time.
The ACS anomaly in 2016 is flagged with an annotation. The Pearson correlation
result (r = -0.46, p = 0.15) is included in the caption to be transparent that
the visual association is not statistically confirmed.

### Figure 5: Eviction rate heatmap (choroplethmapbox)
A choropleth map using Plotly's choroplethmapbox with the official Baltimore CSA
boundary GeoJSON. The colour scale runs from light yellow (low eviction) to deep
red (high eviction). Safe Streets CSAs are outlined in red to show the overlap
between eviction pressure and intervention presence.

### Figure 6: US gun homicides vs Baltimore (dual-axis line)
Places Baltimore's decline in national context. The national CDC data shows a 27%
decline in gun homicides from 2021 to 2024; Baltimore's is 40%. Dual axis because
the scales differ by two orders of magnitude.


## 6. Discussion

### What went well

The narrative structure works. The combination of video clips, data, and text creates
a rhythm that moves the reader through the story without requiring statistical literacy.
The consistent visual language (colours, typography, the teal 2020 line) gives the
site a coherent identity.

The geographic figures are the strongest element. Figure 3 in particular lets the reader
see the story rather than just read it: the persistence of violence in the same
neighbourhoods year after year is immediately legible, and the decline after 2022 is
visible in the thinning of grey pins.

The transparency about data limitations — the 54% vs 40% discrepancy, the estimated
2014-2019 disconnected youth values, the ACS 2016 anomaly, the non-significant
correlation — strengthens rather than weakens the site. It signals to the reader that
the analysis is honest.

### What is missing or could be improved

**Machine learning:** The project does not use ML. A natural extension would be a
spatial clustering model (e.g. DBSCAN on incident coordinates) to identify persistent
hotspots over time and test whether Safe Streets zones overlap with them more than
chance would predict. This was deprioritised in favour of narrative coherence.

**Temporal resolution:** All analysis is annual. Monthly or weekly data would allow
more precise identification of when the decline began and whether it correlates with
specific policy interventions. The Open Baltimore dataset supports this, but the
visualisations would become more complex to interpret.

**Causal identification:** The site is careful to say the data is observational, but it
cannot say more. A proper causal analysis would require a difference-in-differences
design comparing Safe Streets zones to matched control neighbourhoods, similar to the
Webster et al. (2013) study. Our data alone cannot reproduce that design.

**Eviction data is 2023 only.** A time series of eviction rates by CSA would be a
much stronger structural indicator. The current figure shows a snapshot, not a trend.

**The 2016 ACS anomaly** in single-mother household data (32k vs ~65k in adjacent years)
is flagged but not resolved. It may reflect a methodology change in that year's ACS
survey. A more rigorous analysis would verify this with the original ACS microdata.


## 7. Contributions

This is a solo project. All analysis, visualisation, and writing was done by
Benjamin Balslund (s253848).


## 8. References

Segel, E., & Heer, J. (2010). Narrative Visualization: Telling Stories with Data.
*IEEE Transactions on Visualization and Computer Graphics*, 16(6), 1139-1148.

Webster, D. W., Whitehill, J. M., Vernick, J. S., & Curriero, F. C. (2013).
Effects of Baltimore's Safe Streets Program on Gun Violence: A Replication of
Chicago's CeaseFire Program. *Journal of Urban Health*, 90(1), 27-40.
https://pmc.ncbi.nlm.nih.gov/articles/PMC3579298/

Open Baltimore Part 1 Crime Data. Baltimore City. https://data.baltimorecity.gov

Annie E. Casey Foundation KIDS COUNT Data Center. https://datacenter.aecf.org

Federal Reserve Bank of St. Louis (FRED). Baltimore City Disconnected Youth Series.
https://fred.stlouisfed.org/series/B14005DCYACS024510

Gramlich, J. (2026). What the data says about gun deaths in the U.S.
Pew Research Center. https://www.pewresearch.org/short-reads/2026/04/28/what-the-data-says-about-gun-deaths-in-the-us/

Mayor Brandon M. Scott Highlights Historic Reductions in Violent Crime in 2025.
Baltimore City Mayor's Office, January 5, 2026.
https://www.baltimorecity.gov/mayor/news-media/press-releases/2026-01-05-mayor-brandon-m-scott-highlights-historic-reductions-in-violent-crime-in-2025

Baltimore Neighborhood Indicators Alliance. Rate of Evictions per 1,000 Residents.
https://data.baltimorecity.gov

FBI Uniform Crime Reports. Police officer counts via city-data.com.

Callaghan, A. (2024). *Baltimore Streets*. Channel 5.
https://www.youtube.com/watch?v=XQs59YY-e2I
